<a href="https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/shivam25th/flyrank-1st.git"
REPO = Path("/content/flyrank-1st")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )

os.chdir(REPO)

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print("Repository:", REPO)
print("Dataset shape:", df.shape)

Repository: /content/flyrank-1st
Dataset shape: (30000, 44)


# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

### Feature vector

I will use features that are available before the prediction/review decision and that describe the page's content and observed search performance. I will exclude fields that define the target or reveal future outcomes.

The feature vector will contain numeric content/search signals and selected categorical fields. Missing numeric values will be filled using the median, while missing categorical values will be represented as `missing`.

The target is `is_declining_label`, derived from `trend_direction`, so `trend_direction` and `trend_pct` are excluded from the feature vector.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "content_type",
    "main_intent",
    "position_tier",
    "impression_tier"
]

# Keep only columns that actually exist
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

# Numeric missing-value handling
X_numeric = df[numeric_features].copy()
X_numeric = X_numeric.replace([np.inf, -np.inf], np.nan)
X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))

# Categorical missing-value handling
X_categorical = df[categorical_features].copy()
X_categorical = X_categorical.fillna("missing").astype(str)

feature_vector = pd.concat(
    [X_numeric, X_categorical],
    axis=1
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Feature vector shape:", feature_vector.shape)
print("\nMissing values remaining:", feature_vector.isna().sum().sum())

Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_age_days', 'days_since_last_update']
Categorical features: ['content_type', 'main_intent', 'position_tier', 'impression_tier']
Feature vector shape: (30000, 17)

Missing values remaining: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

### Feature availability notes

| Feature group | Meaning | Missing handling | Available before decision? |
|---|---|---|---|
| Search volume | Observed keyword demand signal | Median | Yes |
| Competition / CPC | Search-market characteristics | Median | Yes |
| Word/character count | Content size | Median | Yes |
| Impressions | Observed search exposure | Median | Yes, for the defined observation window |
| CTR | Observed click-through rate | Median | Yes, for the defined observation window |
| Average position | Observed search position | Median | Yes, for the defined observation window |
| Engagement / scroll | Observed user engagement signals | Median | Yes, for the defined observation window |
| Content age / update recency | Content freshness signals | Median | Yes |
| Content type / intent / position tiers | Categorical page/search attributes | `missing` category | Yes |

The important constraint is that a feature must be available at the moment the decision is made. Target-defining or future-outcome fields are excluded even if they are present in the dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature availability check")
print("=" * 40)

for col in feature_vector.columns:
    print(
        f"{col:30s} | "
        f"missing={feature_vector[col].isna().sum():4d} | "
        f"dtype={feature_vector[col].dtype}"
    )

Feature availability check
search_volume                  | missing=   0 | dtype=float64
competition                    | missing=   0 | dtype=float64
cpc                            | missing=   0 | dtype=float64
word_count                     | missing=   0 | dtype=float64
char_count                     | missing=   0 | dtype=float64
impressions_90d                | missing=   0 | dtype=int64
ctr                            | missing=   0 | dtype=float64
avg_position                   | missing=   0 | dtype=float64
engagement_rate                | missing=   0 | dtype=float64
scroll_rate                    | missing=   0 | dtype=float64
ai_traffic_pct                 | missing=   0 | dtype=float64
content_age_days               | missing=   0 | dtype=int64
days_since_last_update         | missing=   0 | dtype=int64
content_type                   | missing=   0 | dtype=object
main_intent                    | missing=   0 | dtype=object
position_tier                  | missing=   0 | dty

## 3. The leakage hunt

### Leakage checks

I checked the feature vector for fields that directly define the target, contain the future outcome, or could reveal information that would not be available when making the decision.

The clearest leakage risk is `trend_direction`, because the target `is_declining_label` is directly derived from it. `trend_pct` is also excluded because it directly describes the observed trend used to define the target.

I also avoid using identifiers such as `content_id` and `client_id` as predictive features. They identify entities rather than describe page behavior and could encourage memorization rather than generalizable learning.

These exclusions are intentional even when the fields are predictive, because predictive power obtained by revealing the answer would be misleading.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

identifier_columns = [
    "content_id",
    "client_id"
]

leakage_candidates = [
    col for col in target_columns + identifier_columns
    if col in df.columns
]

print("Potential leakage / identifier fields:")
for col in leakage_candidates:
    print("-", col)

print("\nLeakage fields present in feature vector:")
print(
    [
        col for col in leakage_candidates
        if col in feature_vector.columns
    ]
)

assert "trend_direction" not in feature_vector.columns
assert "trend_pct" not in feature_vector.columns
assert "is_declining_label" not in feature_vector.columns

print("\nLeakage assertions passed.")




Potential leakage / identifier fields:
- trend_direction
- trend_pct
- is_declining_label
- content_id
- client_id

Leakage fields present in feature vector:
[]

Leakage assertions passed.


## 4. What I excluded and why

### Excluded fields

| Field | Reason for exclusion |
|---|---|
| `trend_direction` | Directly defines the target and would cause target leakage. |
| `trend_pct` | Directly describes the trend outcome used to define the target. |
| `content_id` | Identifier rather than a behavioral feature; could encourage memorization. |
| `client_id` | Identifier/grouping field; should not be used as an ordinary predictive feature. |

I also exclude any feature that is only known after the decision point or that is constructed using the outcome being predicted. The goal is to keep the feature vector usable for honest decision-support rather than maximizing predictive performance through leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = {
    "trend_direction": "Defines the target; target leakage.",
    "trend_pct": "Reveals the observed trend outcome.",
    "content_id": "Identifier; not a behavioral feature.",
    "client_id": "Identifier/grouping field; not an ordinary feature."
}

print("Excluded fields and reasons:\n")

for field, reason in excluded_fields.items():
    if field in df.columns:
        print(f"{field}: {reason}")

Excluded fields and reasons:

trend_direction: Defines the target; target leakage.
trend_pct: Reveals the observed trend outcome.
content_id: Identifier; not a behavioral feature.
client_id: Identifier/grouping field; not an ordinary feature.


- [x] Every section above is filled
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`